# PySpark DataFrame DSL — Part 2
# Select & Filter

## Topics Covered

1. `select()` — Selecting columns
2. `filter()` — Filtering rows
3. `where()` — Filtering rows
4. Conditions and comparison operators
5. Multiple conditions using `&`, `|`, and `~`
6. String conditions
7. `select()` + `filter()` combinations
8. `filter()` vs `where()`
9. Practical Data Engineering examples
10. Quick reference and mental models

---

# 1. `select()` — Selecting Columns

## Definition

`select()` is a DataFrame DSL method used to select specific columns or column expressions from a DataFrame.

It controls **which columns** appear in the resulting DataFrame.

### Basic Syntax

```python
df.select("column_name")
```

Multiple columns:

```python
df.select("column1", "column2", "column3")
```

---

## Example DataFrame

We will use the following DataFrame throughout this section.

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataFrameDSLPart2") \
    .getOrCreate()

data = [
    ("C001", "Karthik", "Chennai", 25, 55000.50),
    ("C002", "Arun", "Mumbai", 28, 62000.75),
    ("C003", "Rahul", "Delhi", 26, 48000.25),
    ("C004", "Priya", "Bangalore", 30, 75000.00),
    ("C005", "Anitha", "Chennai", 24, 45000.00),
    ("C006", "Vijay", "Mumbai", 32, 82000.00)
]

columns = [
    "customer_id",
    "name",
    "city",
    "age",
    "salary"
]

df = spark.createDataFrame(data, columns)

df.show()
```

---

## Example 1 — Select One Column

```python
df.select("name").show()
```

Result:

```text
+-------+
|   name|
+-------+
|Karthik|
|   Arun|
|  Rahul|
|  Priya|
| Anitha|
|  Vijay|
+-------+
```

---

## Example 2 — Select Multiple Columns

```python
df.select(
    "name",
    "city",
    "salary"
).show()
```

Only these columns appear in the resulting DataFrame.

---

## Example 3 — Select Using `col()`

```python
from pyspark.sql.functions import col

df.select(
    col("name"),
    col("city"),
    col("salary")
).show()
```

Using `col()` becomes especially useful when expressions or conditions are involved.

---

## Example 4 — Select a Calculated Column

```python
df.select(
    "name",
    (col("salary") * 12).alias("annual_salary")
).show()
```

Here:

```python
col("salary") * 12
```

is a column expression.

---

## Important Point

`select()` controls the **columns** of the result.

```text
select()
   ↓
Which columns should I return?
```

It does not primarily decide which rows should remain.

For row selection, we use:

```python
filter()
```

or:

```python
where()
```

---

# 2. `filter()` — Filtering Rows

## Definition

`filter()` is a DataFrame DSL method used to keep only rows that satisfy a condition.

### Basic Syntax

```python
df.filter(condition)
```

Think of it as:

> Keep the rows where the condition is TRUE.

---

## Example 1 — Filter by Age

```python
df.filter(
    col("age") > 25
).show()
```

This keeps customers whose age is greater than 25.

Expected matching customers:

```text
Arun
Rahul
Priya
Vijay
```

---

## Example 2 — Filter by Salary

```python
df.filter(
    col("salary") > 60000
).show()
```

This keeps rows where salary is greater than 60,000.

---

## Example 3 — Filter by Exact Value

```python
df.filter(
    col("city") == "Chennai"
).show()
```

This keeps only customers from Chennai.

---

## Example 4 — Filter and Store Result

```python
df2 = df.filter(
    col("age") > 25
)

df2.show()
```

The original:

```python
df
```

is not modified.

---

# 3. `where()` — Filtering Rows

## Definition

`where()` is another DataFrame DSL method used to filter rows.

For DataFrame operations:

```python
filter()
```

and:

```python
where()
```

are equivalent.

---

## Example 1 — `where()`

```python
df.where(
    col("age") > 25
).show()
```

This produces the same result as:

```python
df.filter(
    col("age") > 25
).show()
```

---

## Example 2 — Filter by City

```python
df.where(
    col("city") == "Mumbai"
).show()
```

---

## Example 3 — Filter by Salary

```python
df.where(
    col("salary") >= 70000
).show()
```

---

## `filter()` vs `where()`

There is no meaningful difference in the result for normal DataFrame usage.

```python
df.filter(col("age") > 25)
```

and:

```python
df.where(col("age") > 25)
```

perform the same type of row filtering.

### Mental Model

```text
filter()
   ↓
Keep rows matching condition

where()
   ↓
Keep rows matching condition
```

Choose whichever style is clearer for your code.

---

# 4. Conditions and Comparison Operators

Filtering depends on conditions.

A condition compares values and produces a Boolean expression.

For example:

```python
col("age") > 25
```

Conceptually:

```text
age > 25
   ↓
TRUE / FALSE
   ↓
filter()
   ↓
Keep TRUE rows
```

---

## Comparison Operators

| Operator | Meaning |
|---|---|
| `>` | Greater than |
| `<` | Less than |
| `>=` | Greater than or equal to |
| `<=` | Less than or equal to |
| `==` | Equal to |
| `!=` | Not equal to |

---

## 4.1 Greater Than `>`

```python
df.filter(
    col("age") > 25
).show()
```

Keeps rows where:

```text
age > 25
```

---

## 4.2 Less Than `<`

```python
df.filter(
    col("age") < 26
).show()
```

Keeps rows where:

```text
age < 26
```

---

## 4.3 Greater Than or Equal `>=`

```python
df.filter(
    col("age") >= 25
).show()
```

Keeps rows where:

```text
age >= 25
```

---

## 4.4 Less Than or Equal `<=`

```python
df.filter(
    col("age") <= 25
).show()
```

---

## 4.5 Equal `==`

```python
df.filter(
    col("city") == "Chennai"
).show()
```

Important:

```python
==
```

means comparison.

Do not confuse it with:

```python
=
```

which is assignment in Python.

---

## 4.6 Not Equal `!=`

```python
df.filter(
    col("city") != "Chennai"
).show()
```

This keeps customers whose city is not Chennai.

---

# 5. Multiple Conditions

Real Data Engineering filtering often requires multiple conditions.

For example:

> Customers older than 25 AND salary greater than 60,000.

In PySpark, use:

```python
&
```

for AND.

---

## 5.1 AND — `&`

```python
df.filter(
    (col("age") > 25) &
    (col("salary") > 60000)
).show()
```

Both conditions must be TRUE.

Conceptually:

```text
Condition 1       Condition 2
age > 25    AND   salary > 60000
     |                  |
     +--------+---------+
              |
             TRUE
              |
          Keep row
```

### Important

Put each condition inside parentheses:

```python
(col("age") > 25) &
(col("salary") > 60000)
```

Do not write:

```python
col("age") > 25 & col("salary") > 60000
```

Use parentheses to make the Boolean expression explicit and avoid operator-precedence problems.

---

## 5.2 OR — `|`

Use:

```python
|
```

when either condition can be TRUE.

Example:

```python
df.filter(
    (col("city") == "Chennai") |
    (col("city") == "Mumbai")
).show()
```

This keeps customers from Chennai OR Mumbai.

---

## 5.3 NOT — `~`

Use:

```python
~
```

to negate a condition.

Example:

```python
df.filter(
    ~(col("city") == "Chennai")
).show()
```

This means:

> Keep rows where city is NOT Chennai.

---

# 6. Combining Multiple Conditions

## Example 1 — Age and Salary

```python
df.filter(
    (col("age") >= 25) &
    (col("salary") > 50000)
).show()
```

Meaning:

```text
age >= 25
AND
salary > 50000
```

---

## Example 2 — Two Cities

```python
df.filter(
    (col("city") == "Chennai") |
    (col("city") == "Mumbai")
).show()
```

Meaning:

```text
city = Chennai
OR
city = Mumbai
```

---

## Example 3 — Age Range

```python
df.filter(
    (col("age") >= 25) &
    (col("age") <= 30)
).show()
```

Meaning:

```text
25 <= age <= 30
```

This is useful when filtering a numeric range.

---

## Example 4 — Complex Condition

```python
df.filter(
    (
        (col("age") > 25) &
        (col("salary") > 60000)
    )
    |
    (col("city") == "Chennai")
).show()
```

Meaning:

```text
(age > 25 AND salary > 60000)
OR
city = Chennai
```

---

# 7. String Conditions

Filtering is not limited to numbers.

You can filter text columns.

---

## Example 1 — Exact String Match

```python
df.filter(
    col("city") == "Chennai"
).show()
```

---

## Example 2 — Not Equal to a String

```python
df.filter(
    col("city") != "Chennai"
).show()
```

---

## Example 3 — `startswith()`

You can use string expressions such as:

```python
df.filter(
    col("name").startswith("K")
).show()
```

This keeps names starting with `K`.

---

## Example 4 — `endswith()`

```python
df.filter(
    col("name").endswith("h")
).show()
```

This keeps names ending with `h`.

---

## Example 5 — `contains()`

```python
df.filter(
    col("name").contains("ar")
).show()
```

This checks whether the name contains the specified text.

---

## Example 6 — `like()`

```python
df.filter(
    col("name").like("K%")
).show()
```

`%` represents zero or more characters.

This is similar to SQL:

```sql
WHERE name LIKE 'K%'
```

---

## Example 7 — `rlike()`

`rlike()` allows regular-expression matching.

```python
df.filter(
    col("name").rlike("^K")
).show()
```

This finds names beginning with `K`.

---

# 8. `isin()` — Matching Multiple Values

`isin()` is useful when you want to check whether a column contains one of several values.

Example:

```python
df.filter(
    col("city").isin("Chennai", "Mumbai")
).show()
```

This is conceptually similar to:

```sql
WHERE city IN ('Chennai', 'Mumbai')
```

---

## Example — Multiple Cities

```python
cities = ["Chennai", "Mumbai", "Delhi"]

df.filter(
    col("city").isin(cities)
).show()
```

---

## NOT IN

You can negate `isin()`:

```python
df.filter(
    ~col("city").isin("Chennai", "Mumbai")
).show()
```

This means:

```text
city NOT IN ('Chennai', 'Mumbai')
```

---

# 9. `between()` — Filtering a Range

`between()` is useful for numeric or comparable ranges.

Example:

```python
df.filter(
    col("age").between(25, 30)
).show()
```

This means:

```text
age >= 25
AND
age <= 30
```

Equivalent condition:

```python
df.filter(
    (col("age") >= 25) &
    (col("age") <= 30)
).show()
```

---

## Salary Range

```python
df.filter(
    col("salary").between(50000, 80000)
).show()
```

---

# 10. NULL Conditions

NULL requires special handling.

Do not use:

```python
col("city") == None
```

for normal Spark NULL checks.

Use:

```python
isNull()
```

and:

```python
isNotNull()
```

---

## 10.1 `isNull()`

```python
df.filter(
    col("city").isNull()
).show()
```

This keeps rows where city is NULL.

---

## 10.2 `isNotNull()`

```python
df.filter(
    col("city").isNotNull()
).show()
```

This keeps rows where city is not NULL.

---

# 11. `select()` + `filter()`

This combination is extremely important.

Remember:

```text
select()
   ↓
Choose columns

filter()
   ↓
Choose rows
```

---

## Example 1

```python
df.filter(
    col("age") > 25
).select(
    "name",
    "city",
    "age"
).show()
```

Read it as:

> First keep customers older than 25, then return only name, city, and age.

---

## Example 2 — Salary Filter

```python
df.filter(
    col("salary") > 60000
).select(
    "customer_id",
    "name",
    "salary"
).show()
```

---

## Example 3 — City and Salary

```python
df.filter(
    (col("city") == "Chennai") &
    (col("salary") > 50000)
).select(
    "name",
    "city",
    "salary"
).show()
```

Meaning:

```text
Rows:
    Chennai
    AND
    salary > 50000

Columns:
    name
    city
    salary
```

---

# 12. `select()` Before `filter()` vs `filter()` Before `select()`

Both can be written, but the available columns matter.

---

## Example

```python
df.select(
    "name",
    "city",
    "age"
).filter(
    col("age") > 25
).show()
```

This works because `age` was retained by `select()`.

---

## This Will Fail

```python
df.select(
    "name",
    "city"
).filter(
    col("age") > 25
).show()
```

Why?

After `select()` the resulting DataFrame contains only:

```text
name
city
```

The `age` column no longer exists in that DataFrame.

Then Spark cannot evaluate:

```python
col("age") > 25
```

---

## Safe Mental Model

Before filtering, make sure the columns required by the condition are available.

For example:

```python
df.filter(
    col("age") > 25
).select(
    "name",
    "city"
).show()
```

This is clear:

```text
Original DataFrame
       ↓
Filter using age
       ↓
Select required output columns
```

---

# 13. SQL vs DataFrame DSL

## Select

SQL:

```sql
SELECT name, city
FROM customers;
```

PySpark:

```python
df.select(
    "name",
    "city"
)
```

---

## Filter

SQL:

```sql
SELECT *
FROM customers
WHERE age > 25;
```

PySpark:

```python
df.filter(
    col("age") > 25
)
```

---

## Multiple Conditions

SQL:

```sql
SELECT *
FROM customers
WHERE age > 25
AND salary > 60000;
```

PySpark:

```python
df.filter(
    (col("age") > 25) &
    (col("salary") > 60000)
)
```

---

## OR

SQL:

```sql
SELECT *
FROM customers
WHERE city = 'Chennai'
OR city = 'Mumbai';
```

PySpark:

```python
df.filter(
    (col("city") == "Chennai") |
    (col("city") == "Mumbai")
)
```

---

## IN

SQL:

```sql
SELECT *
FROM customers
WHERE city IN ('Chennai', 'Mumbai');
```

PySpark:

```python
df.filter(
    col("city").isin("Chennai", "Mumbai")
)
```

---

## BETWEEN

SQL:

```sql
SELECT *
FROM customers
WHERE age BETWEEN 25 AND 30;
```

PySpark:

```python
df.filter(
    col("age").between(25, 30)
)
```

---

# 14. Practical Data Engineering Examples

## Example 1 — High Salary Customers

Requirement:

> Find customers earning more than 60,000.

```python
df.filter(
    col("salary") > 60000
).show()
```

---

## Example 2 — Chennai Customers

Requirement:

> Find all customers from Chennai.

```python
df.filter(
    col("city") == "Chennai"
).show()
```

---

## Example 3 — Chennai Customers With Salary > 50,000

```python
df.filter(
    (col("city") == "Chennai") &
    (col("salary") > 50000)
).show()
```

---

## Example 4 — Customers From Selected Cities

```python
df.filter(
    col("city").isin(
        "Chennai",
        "Mumbai",
        "Delhi"
    )
).show()
```

---

## Example 5 — Customers Between 25 and 30

```python
df.filter(
    col("age").between(25, 30)
).show()
```

---

## Example 6 — Select Only Required Columns

```python
df.filter(
    col("salary") > 60000
).select(
    "customer_id",
    "name",
    "salary"
).show()
```

---

## Example 7 — Filter + Calculate + Select

```python
df.filter(
    col("salary") > 60000
).select(
    "name",
    "city",
    col("salary").alias("monthly_salary"),
    (col("salary") * 12).alias("annual_salary")
).show()
```

This combines several DataFrame DSL concepts:

```text
filter()
    ↓
Keep required rows

select()
    ↓
Choose required columns

col()
    ↓
Reference columns

alias()
    ↓
Name expressions
```

---

# 15. Multiple Filters

You can chain filters.

Example:

```python
df.filter(
    col("age") > 25
).filter(
    col("salary") > 60000
).show()
```

Conceptually:

```text
Original DataFrame
       ↓
age > 25
       ↓
salary > 60000
       ↓
Final DataFrame
```

You can also combine them into one condition:

```python
df.filter(
    (col("age") > 25) &
    (col("salary") > 60000)
).show()
```

Both express the same logical requirement.

For readability, use whichever form makes the transformation easier to understand.

---

# 16. `filter()` With a String Condition

PySpark also allows a SQL-style string condition.

Example:

```python
df.filter("age > 25").show()
```

Another:

```python
df.filter(
    "salary > 60000"
).show()
```

Multiple conditions:

```python
df.filter(
    "age > 25 AND salary > 60000"
).show()
```

City:

```python
df.filter(
    "city = 'Chennai'"
).show()
```

This is different from the column-expression style:

```python
df.filter(
    col("age") > 25
).show()
```

Both are valid DataFrame DSL approaches.

---

# 17. Column Expression vs String Condition

## Column Expression

```python
df.filter(
    col("age") > 25
)
```

Advantages:

- Python expression style
- Works naturally with `col()`
- Easy to combine with other DataFrame expressions
- Useful for programmatically building transformations

---

## String Condition

```python
df.filter(
    "age > 25"
)
```

Advantages:

- Looks similar to SQL
- Can be convenient for simple conditions

---

## Complex Example

Column expression:

```python
df.filter(
    (col("age") > 25) &
    (col("salary") > 60000)
)
```

String condition:

```python
df.filter(
    "age > 25 AND salary > 60000"
)
```

Both express the same logical filtering requirement.

---

# 18. Important Difference: `select()` vs `filter()`

This is one of the most important concepts.

## `select()`

Controls:

> **Which columns?**

Example:

```python
df.select(
    "name",
    "city"
)
```

Result:

```text
name
city
```

---

## `filter()`

Controls:

> **Which rows?**

Example:

```python
df.filter(
    col("age") > 25
)
```

Result:

```text
Only rows where age > 25
```

---

## Combined

```python
df.filter(
    col("age") > 25
).select(
    "name",
    "city"
)
```

Meaning:

```text
FILTER → Which rows?
SELECT → Which columns?
```

### Mental Model

```text
                 DataFrame
                     |
             +-------+-------+
             |               |
             ↓               ↓
          filter()        select()
             |               |
        Which rows?      Which columns?
             |               |
             +-------+-------+
                     |
                     ↓
              Final DataFrame
```

---

# 19. Important Rules to Remember

## Rule 1 — Use `==` for equality

Correct:

```python
col("city") == "Chennai"
```

Not:

```python
col("city") = "Chennai"
```

---

## Rule 2 — Use `&` for AND

Correct:

```python
(col("age") > 25) & (col("salary") > 60000)
```

---

## Rule 3 — Use `|` for OR

Correct:

```python
(col("city") == "Chennai") | (col("city") == "Mumbai")
```

---

## Rule 4 — Use `~` for NOT

```python
~(col("city") == "Chennai")
```

---

## Rule 5 — Use parentheses around conditions

Correct:

```python
(col("age") > 25) & (col("salary") > 60000)
```

This makes the Boolean logic explicit and avoids operator-precedence problems.

---

## Rule 6 — Use `isNull()` / `isNotNull()` for NULL

```python
col("city").isNull()
```

```python
col("city").isNotNull()
```

---

# 20. Quick Reference

| Operation | Purpose | Example |
|---|---|---|
| `select()` | Select columns | `df.select("name")` |
| `filter()` | Filter rows | `df.filter(col("age") > 25)` |
| `where()` | Filter rows | `df.where(col("age") > 25)` |
| `>` | Greater than | `col("age") > 25` |
| `<` | Less than | `col("age") < 25` |
| `>=` | Greater/equal | `col("age") >= 25` |
| `<=` | Less/equal | `col("age") <= 25` |
| `==` | Equal | `col("city") == "Chennai"` |
| `!=` | Not equal | `col("city") != "Chennai"` |
| `&` | AND | `(condition1) & (condition2)` |
| `|` | OR | `(condition1) | (condition2)` |
| `~` | NOT | `~condition` |
| `isin()` | Match multiple values | `col("city").isin(...)` |
| `between()` | Range filtering | `col("age").between(25, 30)` |
| `isNull()` | Find NULL values | `col("city").isNull()` |
| `isNotNull()` | Find non-NULL values | `col("city").isNotNull()` |
| `startswith()` | String prefix | `col("name").startswith("K")` |
| `endswith()` | String suffix | `col("name").endswith("h")` |
| `contains()` | String contains | `col("name").contains("ar")` |
| `like()` | SQL-style pattern | `col("name").like("K%")` |
| `rlike()` | Regex matching | `col("name").rlike("^K")` |

---

# 21. Final Mental Model

```text
                    DataFrame
                        |
          +-------------+-------------+
          |                           |
          ↓                           ↓
      filter()                    select()
          |                           |
     Which rows?                Which columns?
          |                           |
          ↓                           ↓
    Conditions                  Column expressions
          |                           |
     +----+----+                 +----+----+
     |    |    |                 |    |    |
     >    &    |                 col  alias calculations
     <    |    ~
     >=   |
     <=   |
     ==   |
     !=   |
          |
          ↓
    Filtered DataFrame
          |
          ↓
    Selected DataFrame
```

---

# Key Takeaways

1. `select()` controls **which columns** are returned.
2. `filter()` controls **which rows** are retained.
3. `where()` is an alternative to `filter()`.
4. `col()` is commonly used to build column-based conditions.
5. `>` `<` `>=` `<=` `==` and `!=` are comparison operators.
6. `&` represents AND.
7. `|` represents OR.
8. `~` represents NOT.
9. Use parentheses around individual conditions.
10. `isin()` is useful for multiple possible values.
11. `between()` is useful for ranges.
12. Use `isNull()` and `isNotNull()` for NULL handling.
13. String methods such as `startswith()`, `contains()`, `like()`, and `rlike()` are useful for text filtering.
14. `select()` and `filter()` can be chained to build DataFrame DSL transformation pipelines.
15. DataFrame transformations return new DataFrames rather than modifying the original DataFrame.

## Core Pattern

```python
from pyspark.sql.functions import col

result = df \
    .filter(
        (col("age") > 25) &
        (col("salary") > 60000)
    ) \
    .select(
        "customer_id",
        "name",
        "city",
        "salary"
    )

result.show()
```

Read this as:

```text
DataFrame
    ↓
Filter rows
    ↓
Select required columns
    ↓
Final DataFrame
```

This is a fundamental pattern in PySpark DataFrame DSL and appears frequently in real-world ETL and Data Engineering pipelines.
